# 02 — Baseline con Machine Learning Clásico

Objetivo: establecer un *techo de rendimiento* con modelos clásicos antes de pasar a Deep Learning.

1. **Features**: HOG, píxeles aplanados, embeddings de ResNet18 congelada
2. **Modelos**: SVM (linear, RBF), Random Forest, KNN, con 5-fold CV
3. **Desbalance**: comparar con y sin `class_weight='balanced'`
4. **Métricas**: accuracy, F1 macro/weighted, confusion matrix
5. **Conclusiones**: techo del ML clásico y por qué necesitamos DL

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch, numpy as np, random
torch.manual_seed(42); np.random.seed(42); random.seed(42)

METADATA_PATH = ROOT / 'data' / 'metadata.csv'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

import pandas as pd, matplotlib.pyplot as plt
from PIL import Image
df = pd.read_csv(METADATA_PATH)
print(f'Imágenes: {len(df)}  Clases: {df["compound_id"].nunique()}')

Device: cuda


Imágenes: 58800  Clases: 196


## 2.1 Extracción de features

In [2]:
from skimage.feature import hog
from tqdm.auto import tqdm

IMG_SIZE = 64  # reducimos resolución para que ML clásico sea viable

def load_gray(fp):
    return np.array(Image.open(ROOT / fp).convert('L').resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0

def feat_pixels(im):
    return im.flatten()

def feat_hog(im):
    return hog(im, orientations=8, pixels_per_cell=(8, 8), cells_per_block=(2, 2), feature_vector=True)

subset = df.copy()
if len(subset) > 4000:
    subset = subset.groupby('compound_id', group_keys=False).apply(lambda g: g.head(min(40, len(g))))
print(f'Subset para ML: {len(subset)} imágenes')

images = [load_gray(fp) for fp in tqdm(subset['filepath'])]
X_pix = np.stack([feat_pixels(im) for im in images])
X_hog = np.stack([feat_hog(im) for im in images])
y = subset['compound_id'].values
splits = subset['split'].values
print(f'X_pix shape: {X_pix.shape}, X_hog shape: {X_hog.shape}')

Subset para ML: 7840 imágenes


C:\Users\motam\AppData\Local\Temp\ipykernel_19084\53309091.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset = subset.groupby('compound_id', group_keys=False).apply(lambda g: g.head(min(40, len(g))))


  0%|          | 0/7840 [00:00<?, ?it/s]

X_pix shape: (7840, 4096), X_hog shape: (7840, 1568)


In [3]:
import torch
from torchvision import models, transforms

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = torch.nn.Identity()
resnet.eval().to(DEVICE)

X_cnn = []
with torch.no_grad():
    for fp in tqdm(subset['filepath']):
        im = Image.open(ROOT / fp).convert('RGB')
        x = tf(im).unsqueeze(0).to(DEVICE)
        f = resnet(x).cpu().numpy().squeeze()
        X_cnn.append(f)
X_cnn = np.stack(X_cnn)
print(f'X_cnn shape: {X_cnn.shape}')

  0%|          | 0/7840 [00:00<?, ?it/s]

X_cnn shape: (7840, 512)


## 2.2 Comparación de modelos (5-fold CV en train)

In [4]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

train_mask = splits == 'train'

feature_sets = {'pixels': X_pix, 'hog': X_hog, 'cnn_embed': X_cnn}
models_def = {
    'SVM-linear':  lambda: Pipeline([('s', StandardScaler()), ('m', SVC(kernel='linear', class_weight='balanced'))]),
    'SVM-rbf':     lambda: Pipeline([('s', StandardScaler()), ('m', SVC(kernel='rbf', class_weight='balanced'))]),
    'RandomForest':lambda: RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'KNN':         lambda: Pipeline([('s', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=5))]),
}

rows = []
for feat_name, X in feature_sets.items():
    Xt = X[train_mask]; yt = y[train_mask]
    for mname, mfn in models_def.items():
        try:
            scores = cross_val_score(mfn(), Xt, yt, cv=3, scoring='accuracy', n_jobs=-1)
            rows.append({'features': feat_name, 'model': mname, 'cv_acc_mean': scores.mean(), 'cv_acc_std': scores.std()})
            print(f'{feat_name:10s} {mname:13s}  acc={scores.mean():.3f} ± {scores.std():.3f}')
        except Exception as e:
            print(f'{feat_name:10s} {mname:13s}  ERROR: {e}')
cv_df = pd.DataFrame(rows)
display(cv_df.sort_values('cv_acc_mean', ascending=False))

pixels     SVM-linear     acc=0.206 ± 0.019


pixels     SVM-rbf        acc=0.098 ± 0.013


pixels     RandomForest   acc=0.189 ± 0.017


pixels     KNN            acc=0.072 ± 0.009


hog        SVM-linear     acc=0.468 ± 0.004


...

cnn_embed  SVM-linear     acc=0.628 ± 0.005


cnn_embed  SVM-rbf        acc=0.493 ± 0.003


cnn_embed  RandomForest   acc=0.464 ± 0.006


cnn_embed  KNN            acc=0.324 ± 0.011


,features,model,cv_acc_mean,cv_acc_std
8,cnn_embed,SVM-linear,0.628090,0.005071
9,cnn_embed,SVM-rbf,0.492551,0.003491
4,hog,SVM-linear,0.467841,0.003802
10,cnn_embed,RandomForest,0.464209,0.005580
6,hog,RandomForest,0.416788,0.001899
5,hog,SVM-rbf,0.359009,0.010631
11,cnn_embed,KNN,0.323944,0.011215
7,hog,KNN,0.253634,0.004472
0,pixels,SVM-linear,0.205847,0.018552
2,pixels,RandomForest,0.188950,0.017235


## 2.3 Mejor modelo: evaluación en test

In [5]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

best_row = cv_df.sort_values('cv_acc_mean', ascending=False).iloc[0]
feat_name, mname = best_row['features'], best_row['model']
print(f'Mejor modelo CV: {mname} con features {feat_name}')

X = feature_sets[feat_name]
model = models_def[mname]()
model.fit(X[splits == 'train'], y[splits == 'train'])
y_pred = model.predict(X[splits == 'test'])
y_true = y[splits == 'test']

print(f'\nAccuracy en test: {(y_true == y_pred).mean():.3f}')
print(f'F1 macro:       {f1_score(y_true, y_pred, average="macro", zero_division=0):.3f}')
print(f'F1 weighted:    {f1_score(y_true, y_pred, average="weighted", zero_division=0):.3f}')

Mejor modelo CV: SVM-linear con features cnn_embed



Accuracy en test: 0.690
F1 macro:       0.683
F1 weighted:    0.690


## 2.4 Conclusiones

- **Techo del ML clásico**: las features hand-crafted (HOG, píxeles) son insuficientes para distinguir ~200 clases con buena precisión. Los embeddings de una CNN preentrenada elevan el rendimiento, pero seguimos limitados por usar la red como extractor fijo.
- **Por qué pasar a DL**: la información discriminante (anillos, dobles enlaces, posiciones de heteroátomos) requiere un detector entrenado *específicamente* para esta tarea. Los siguientes notebooks (03, 04) entrenan CNNs y transfer learning para superar este baseline.